# Data Ingestion, Cleaning & Preprocessing with Pandas
## Retail Orders Dataset (provided workbook)

This notebook cleans the uploaded `retail-orders-raw.csv.xlsx` file. It retains the original business fields, removes spreadsheet artifacts and duplicate rows, validates dates and numeric fields, imputes missing/invalid values, creates sales features, and exports `cleaned_retail_orders.csv`.

**Important source limitations:** The uploaded workbook contains only 12 data rows (11 after exact duplicate removal), not 10,000+. It also has no unit cost or profit field, so profit margin cannot be calculated honestly. No artificial rows or costs are invented. Use a larger approved dataset and add actual cost data if your assignment strictly requires 10,000+ rows and profit margin.


## 1. Import libraries and load the supplied Excel file

In [ ]:
import pandas as pd
import numpy as np

raw_all = pd.read_excel("retail-orders-raw.csv.xlsx")
print("Original shape:", raw_all.shape)
display(raw_all.head(10))


## 2. Select meaningful columns and inspect data quality

In [ ]:
expected = [
    "order_id", "order_date", "customer_segment", "city", "category",
    "quantity", "unit_price", "discount_pct", "payment_status"
]
raw = raw_all[expected].copy()  # Drops empty Unnamed columns / unlabelled spreadsheet artifact

print("Rows:", len(raw))
print("Data types:")
display(raw.dtypes.to_frame("dtype"))
print("Missing values:")
display(raw.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", raw.duplicated().sum())
display(raw)


## 3. Remove duplicate records and convert data types

In [ ]:
df = raw.drop_duplicates().copy()

def parse_order_date(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (int, float, np.integer, np.floating)):
        # Excel date serial number conversion
        return pd.to_datetime(value, unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(value, errors="coerce")

df["order_date"] = df["order_date"].apply(parse_order_date)
df["date_imputed"] = df["order_date"].isna()

for col in ["quantity", "unit_price", "discount_pct"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows after duplicate removal:", len(df))
display(df.dtypes.to_frame("dtype_after_conversion"))


## 4. Validate values and handle missing data

In [ ]:
# Business-rule validation
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["unit_price"] <= 0, "unit_price"] = np.nan
df.loc[(df["discount_pct"] < 0) | (df["discount_pct"] > 100), "discount_pct"] = np.nan

# Normalize text labels
for col in ["order_id", "customer_segment", "city", "category", "payment_status"]:
    df[col] = df[col].astype("string").str.strip()

df["customer_segment"] = df["customer_segment"].str.title()
df["city"] = df["city"].str.title()
df["category"] = df["category"].str.title()
df["payment_status"] = df["payment_status"].str.title()

# Categorical missing values -> mode
for col in ["customer_segment", "city", "category", "payment_status"]:
    mode = df[col].mode(dropna=True)
    df[col] = df[col].fillna(mode.iloc[0] if len(mode) else "Unknown")

# Numeric missing/invalid values -> category median, then overall median
for col in ["quantity", "unit_price", "discount_pct"]:
    overall_median = df[col].median()
    if pd.isna(overall_median):
        overall_median = 0
    category_medians = df.groupby("category")[col].transform("median")
    df[col] = df[col].fillna(category_medians).fillna(overall_median)

valid_dates = df["order_date"].dropna().sort_values()
median_date = valid_dates.iloc[len(valid_dates)//2]
df["order_date"] = df["order_date"].fillna(median_date)

df["quantity"] = df["quantity"].round().astype("int64")
df["unit_price"] = df["unit_price"].round(2)
df["discount_pct"] = df["discount_pct"].round(2)

display(df.isna().sum().to_frame("missing_after_cleaning"))


## 5. Outlier and invalid-value treatment

This small sample has only 11 unique orders, so statistical IQR outlier capping would not be reliable. Instead, this notebook applies defensible business constraints: quantity must be positive, unit price must be positive, and discount must be between 0% and 100%. Values violating those rules are treated as invalid and imputed. For a 10,000+ row dataset, add a documented IQR/percentile review with the business owner.

In [ ]:
print("Quantity range:", df["quantity"].min(), "to", df["quantity"].max())
print("Unit price range:", df["unit_price"].min(), "to", df["unit_price"].max())
print("Discount range:", df["discount_pct"].min(), "to", df["discount_pct"].max())


## 6. Feature engineering

In [ ]:
df["gross_sales"] = (df["quantity"] * df["unit_price"]).round(2)
df["discount_amount"] = (df["gross_sales"] * df["discount_pct"] / 100).round(2)
df["net_sales"] = (df["gross_sales"] - df["discount_amount"]).round(2)
df["order_month"] = df["order_date"].dt.month.astype("int64")
df["order_year"] = df["order_date"].dt.year.astype("int64")

# Profit margin is not calculable: the source workbook has no unit_cost or profit column.
print("Profit margin omitted because unit cost/profit data is not present in the source.")
display(df.head())


## 7. Validate and export the cleaned CSV

In [ ]:
clean = df[[
    "order_id", "order_date", "order_month", "order_year",
    "customer_segment", "city", "category", "quantity", "unit_price",
    "discount_pct", "payment_status", "date_imputed",
    "gross_sales", "discount_amount", "net_sales"
]].copy()

assert clean.isna().sum().sum() == 0
assert not clean.duplicated().any()
assert clean["quantity"].gt(0).all()
assert clean["unit_price"].gt(0).all()
assert clean["discount_pct"].between(0, 100).all()

clean["order_date"] = clean["order_date"].dt.strftime("%Y-%m-%d")
clean.to_csv("cleaned_retail_orders.csv", index=False)

print("Clean dataset shape:", clean.shape)
print("Missing cells:", clean.isna().sum().sum())
print("Duplicate rows:", clean.duplicated().sum())
display(clean)


## 8. Before-and-after summary and conclusion

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Rows", "Exact duplicate rows", "Missing cells"],
    "Before": [len(raw), int(raw.duplicated().sum()), int(raw.isna().sum().sum())],
    "After": [len(clean), int(clean.duplicated().sum()), int(clean.isna().sum().sum())]
})
display(summary)


### Conclusion

The supplied retail-orders workbook was ingested with Pandas, empty spreadsheet columns were excluded, one exact duplicate was removed, Excel serial and invalid text dates were parsed, invalid quantity/price/discount values were handled, missing values were imputed, labels were standardized, and month/year plus gross sales, discount amount and net sales were engineered.

**Assignment gap to disclose:** This source contains 12 rows rather than 10,000+, and it does not contain unit cost or profit data. Therefore, profit margin cannot be derived. Do not claim otherwise in the submission.
